In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Directory containing all saved OOF predictions
prediction_dir = Path("../../../data/processed/model_predictions")

# All trained model versions
model_files = {
    "V1.1": "v1_1_predictions.csv",
    "V1.2": "v1_2_predictions.csv",

    "V2.1": "v2_1_predictions.csv",
    "V2.2": "v2_2_predictions.csv",
    "V2.3": "v2_3_predictions.csv",
    "V2.4": "v2_4_predictions.csv",

    "V3.1": "v3_1_predictions.csv",
    "V3.2": "v3_2_predictions.csv",
    "V3.3": "v3_3_predictions.csv",
    "V3.4": "v3_4_predictions.csv",

    "V4.1": "v4_1_predictions.csv",
    "V4.2": "v4_2_predictions.csv",
    "V4.3": "v4_3_predictions.csv"
}

# Load predictions into a dictionary
predictions = {}

for model_name, filename in model_files.items():
    file_path = prediction_dir / filename
    predictions[model_name] = pd.read_csv(file_path)

print("Successfully loaded prediction files:", len(predictions))

for model_name, df in predictions.items():
    print(f"{model_name}: {df.shape}")

Successfully loaded prediction files: 13
V1.1: (219983, 6)
V1.2: (219983, 6)
V2.1: (219983, 6)
V2.2: (219983, 6)
V2.3: (219983, 6)
V2.4: (219983, 6)
V3.1: (219983, 6)
V3.2: (219983, 6)
V3.3: (219983, 6)
V3.4: (219983, 6)
V4.1: (219983, 6)
V4.2: (219983, 6)
V4.3: (219983, 6)


In [2]:
# Use V4.3 as the reference
reference = predictions["V4.3"]

reference = reference.sort_values(
    ["user_id", "order_id"]
).reset_index(drop=True)

for model_name, df in predictions.items():

    df_sorted = df.sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)

    same_interactions = (
        df_sorted[["user_id", "order_id"]].equals(
            reference[["user_id", "order_id"]]
        )
    )

    same_labels = df_sorted["y_true"].equals(
        reference["y_true"]
    )

    same_folds = df_sorted["fold"].equals(
        reference["fold"]
    )

    no_missing = df_sorted.isnull().sum().sum() == 0

    no_duplicates = not df_sorted.duplicated(
        subset=["user_id", "order_id"]
    ).any()

    valid_probabilities = df_sorted["y_pred_proba"].between(
        0, 1
    ).all()

    print(f"\n{model_name}")
    print("Same interactions:", same_interactions)
    print("Same actual labels:", same_labels)
    print("Same folds:", same_folds)
    print("No missing values:", no_missing)
    print("No duplicate interactions:", no_duplicates)
    print("Valid probabilities:", valid_probabilities)


V1.1
Same interactions: True
Same actual labels: True
Same folds: True
No missing values: True
No duplicate interactions: True
Valid probabilities: True

V1.2
Same interactions: True
Same actual labels: True
Same folds: True
No missing values: True
No duplicate interactions: True
Valid probabilities: True

V2.1
Same interactions: True
Same actual labels: True
Same folds: True
No missing values: True
No duplicate interactions: True
Valid probabilities: True

V2.2
Same interactions: True
Same actual labels: True
Same folds: True
No missing values: True
No duplicate interactions: True
Valid probabilities: True

V2.3
Same interactions: True
Same actual labels: True
Same folds: True
No missing values: True
No duplicate interactions: True
Valid probabilities: True

V2.4
Same interactions: True
Same actual labels: True
Same folds: True
No missing values: True
No duplicate interactions: True
Valid probabilities: True

V3.1
Same interactions: True
Same actual labels: True
Same folds: True
No m

In [3]:
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score
)

# Store overall and fold-wise evaluation results
overall_results = []
fold_results = []

for model_name, df in predictions.items():

    y_true = df["y_true"]
    y_prob = df["y_pred_proba"]

    # Pooled OOF metrics
    overall_results.append({
        "Model": model_name,
        "Log Loss": log_loss(y_true, y_prob),
        "Brier Score": brier_score_loss(y_true, y_prob),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    })

    # Calculate metrics separately for each fold
    for fold in sorted(df["fold"].unique()):

        fold_df = df[df["fold"] == fold]

        fold_results.append({
            "Model": model_name,
            "Fold": fold,
            "Log Loss": log_loss(
                fold_df["y_true"],
                fold_df["y_pred_proba"]
            ),
            "Brier Score": brier_score_loss(
                fold_df["y_true"],
                fold_df["y_pred_proba"]
            ),
            "ROC-AUC": roc_auc_score(
                fold_df["y_true"],
                fold_df["y_pred_proba"]
            )
        })

# Convert results into DataFrames
overall_results_df = pd.DataFrame(overall_results)
fold_results_df = pd.DataFrame(fold_results)

# Sort by primary metric
overall_results_df = overall_results_df.sort_values(
    "Log Loss"
).reset_index(drop=True)

print("POOLED OOF RESULTS")
display(overall_results_df.round(5))

print("\nFOLD-WISE RESULTS")
display(fold_results_df.round(5))

POOLED OOF RESULTS


,Model,Log Loss,Brier Score,ROC-AUC
0,V2.4,0.59034,0.20208,0.68621
1,V2.2,0.59063,0.20211,0.68506
2,V2.3,0.59084,0.20228,0.68503
3,V2.1,0.59117,0.20233,0.68372
4,V1.2,0.59418,0.20338,0.67396
5,V1.1,0.59435,0.20347,0.67391
6,V4.3,0.61499,0.21271,0.63563
7,V4.2,0.61532,0.21285,0.63480
8,V4.1,0.61534,0.21286,0.63488
9,V3.2,0.61696,0.21365,0.62671



FOLD-WISE RESULTS


,Model,Fold,Log Loss,Brier Score,ROC-AUC
0,V1.1,1,0.59453,0.20362,0.68593
1,V1.1,2,0.59009,0.20138,0.66226
2,V1.1,3,0.60450,0.20814,0.66989
3,V1.1,4,0.59491,0.20368,0.66497
4,V1.1,5,0.58603,0.19975,0.68200
...,...,...,...,...,...
60,V4.3,1,0.61787,0.21409,0.63887
61,V4.3,2,0.60768,0.20922,0.62811
62,V4.3,3,0.62596,0.21780,0.63529
63,V4.3,4,0.61351,0.21206,0.63203


In [4]:
# Summarise fold-wise performance for every model
fold_summary = (
    fold_results_df
    .groupby("Model")
    .agg(
        Mean_Log_Loss=("Log Loss", "mean"),
        Std_Log_Loss=("Log Loss", "std"),
        Mean_Brier=("Brier Score", "mean"),
        Std_Brier=("Brier Score", "std"),
        Mean_ROC_AUC=("ROC-AUC", "mean"),
        Std_ROC_AUC=("ROC-AUC", "std")
    )
    .reset_index()
)

# Sort using mean fold Log Loss
fold_summary = fold_summary.sort_values(
    "Mean_Log_Loss"
).reset_index(drop=True)

display(fold_summary.round(5))

,Model,Mean_Log_Loss,Std_Log_Loss,Mean_Brier,Std_Brier,Mean_ROC_AUC,Std_ROC_AUC
0,V2.4,0.59011,0.00511,0.20197,0.00227,0.68610,0.00605
1,V2.2,0.59038,0.00530,0.20200,0.00236,0.68494,0.00588
2,V2.3,0.59060,0.00521,0.20218,0.00232,0.68489,0.00584
3,V2.1,0.59092,0.00542,0.20222,0.00242,0.68359,0.00568
4,V1.2,0.59384,0.00680,0.20322,0.00312,0.67304,0.01033
5,V1.1,0.59401,0.00689,0.20332,0.00316,0.67301,0.01046
6,V4.3,0.61463,0.00759,0.21254,0.00354,0.63602,0.00676
7,V4.2,0.61498,0.00754,0.21269,0.00352,0.63510,0.00645
8,V4.1,0.61499,0.00749,0.21270,0.00351,0.63520,0.00647
9,V3.2,0.61657,0.00770,0.21347,0.00354,0.62601,0.01320



## 2. Selection of Representative Models from Each Information Group

### Objective

The objective of this experiment is to select one representative model from each information group. Each group captures a different source of information that may help predict a student's next question outcome.

We compare the candidate models using:

- **Log Loss (Primary):** Evaluates the quality of predicted probabilities.
- **Brier Score (Secondary):** Measures the squared error of predicted probabilities.
- **ROC-AUC (Supporting):** Evaluates the model's ability to distinguish between the two outcome classes.
- **Fold-wise stability:** Examines whether performance is consistent across the five validation folds.

### Group 1 — Overall Student History

**Selected model: V1.1**

V1.2 incorporates the logarithm of previous student interactions in addition to historical accuracy. Although V1.2 achieves marginally better metrics, the improvement is negligible.

V1.1 is selected because it provides a simpler representation of overall student performance with practically equivalent predictive performance.

### Group 2 — Same-Skill Student History

**Selected model: V2.4**

V2.4 uses fractional same-skill historical accuracy along with fractional interaction experience.

It achieves the lowest Log Loss, Brier Score and highest ROC-AUC among the Group 2 candidates. Fractional weighting allows previous questions to contribute according to their skill overlap with the target question.

### Group 3 — Other-Skill Student History

**Selected model: V3.2**

V3.2 uses fractional other-skill historical accuracy.

V3.2 achieves the lowest Log Loss and highest ROC-AUC in Group 3. V3.4 achieves a marginally lower Brier Score, but the difference is negligible. The additional experience feature in V3.4 provides almost no improvement.

### Group 4 — Question Difficulty

**Selected model: V4.3**

V4.3 incorporates question smoothed accuracy, previous question exposure and answer type.

It achieves the best overall performance among the Group 4 candidates across the three reported metrics.

### Final Selection

| Information Group | Selected Model | Selected Representation |
|---|---|---|
| G1: Overall Student History | V1.1 | Overall smoothed accuracy |
| G2: Same-Skill History | V2.4 | Fractional accuracy + experience |
| G3: Other-Skill History | V3.2 | Fractional other-skill accuracy |
| G4: Question Difficulty | V4.3 | Question accuracy + exposure + answer type |

### Next Experiment

The selected representations will be combined into a single Logistic Regression model.

The objective is to investigate whether these four information sources provide complementary predictive information and improve performance compared with the individual models.


In [5]:
# Selected representative models
selected_models = {
    "G1": "V1.1",
    "G2": "V2.4",
    "G3": "V3.2",
    "G4": "V4.3"
}

# Features contributed by each information group
selected_features = {
    "G1": [
        "overall_smoothed_accuracy"
    ],

    "G2": [
        "weighted_same_skill_smoothed_accuracy",
        "log_weighted_same_skill_previous_interactions"
    ],

    "G3": [
        "weighted_other_skill_smoothed_accuracy"
    ],

    "G4": [
        "question_smoothed_accuracy",
        "log_question_previous_interactions",
        "answer_type"
    ]
}

print("Selected models:")
for group, model in selected_models.items():
    print(f"{group}: {model}")

print("\nSelected features:")
for group, features in selected_features.items():
    print(f"{group}: {features}")

Selected models:
G1: V1.1
G2: V2.4
G3: V3.2
G4: V4.3

Selected features:
G1: ['overall_smoothed_accuracy']
G2: ['weighted_same_skill_smoothed_accuracy', 'log_weighted_same_skill_previous_interactions']
G3: ['weighted_other_skill_smoothed_accuracy']
G4: ['question_smoothed_accuracy', 'log_question_previous_interactions', 'answer_type']


In [6]:
from pathlib import Path

feature_dir = Path(
    "../../../data/processed/engineered_features"
)

# Display all saved feature CSV files
for file in sorted(feature_dir.rglob("*.csv")):
    print(file.relative_to(feature_dir))

group_1_overall_history\v1.1\fold_1_train.csv
group_1_overall_history\v1.1\fold_1_validation.csv
group_1_overall_history\v1.1\fold_2_train.csv
group_1_overall_history\v1.1\fold_2_validation.csv
group_1_overall_history\v1.1\fold_3_train.csv
group_1_overall_history\v1.1\fold_3_validation.csv
group_1_overall_history\v1.1\fold_4_train.csv
group_1_overall_history\v1.1\fold_4_validation.csv
group_1_overall_history\v1.1\fold_5_train.csv
group_1_overall_history\v1.1\fold_5_validation.csv
group_1_overall_history\v1.2\fold_1_train.csv
group_1_overall_history\v1.2\fold_1_validation.csv
group_1_overall_history\v1.2\fold_2_train.csv
group_1_overall_history\v1.2\fold_2_validation.csv
group_1_overall_history\v1.2\fold_3_train.csv
group_1_overall_history\v1.2\fold_3_validation.csv
group_1_overall_history\v1.2\fold_4_train.csv
group_1_overall_history\v1.2\fold_4_validation.csv
group_1_overall_history\v1.2\fold_5_train.csv
group_1_overall_history\v1.2\fold_5_validation.csv
group_2_same_skill_history\v2.